# RCC Subtype Classification - Feature Extraction
### BME 515 Final Project
Extract patch-level features using pretrained ResNet50, save as .pt files for CLAM input

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from pathlib import Path
import numpy as np

BASE_DIR = Path('/Users/zhangruikun/Desktop/BME515')
PATCH_DIR = BASE_DIR / 'patches'
FEATURE_DIR = BASE_DIR / 'features'
FEATURE_DIR.mkdir(exist_ok=True)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# load pretrained ResNet50, remove final classification layer
# output will be 2048-dim feature vector per patch
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model = torch.nn.Sequential(*list(model.children())[:-1])  # remove fc layer
model = model.to(device)
model.eval()
print('Model ready')

In [ ]:
# standard ImageNet normalization
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
def extract_slide_features(slide_name, batch_size=32):
    """
    Load all patches for a slide, run through ResNet50,
    return feature matrix of shape [N, 2048]
    """
    patch_dir = PATCH_DIR / slide_name
    patch_paths = sorted(patch_dir.glob('*.png'))

    if len(patch_paths) == 0:
        print(f'No patches found for {slide_name}')
        return None

    all_features = []

    with torch.no_grad():
        for i in range(0, len(patch_paths), batch_size):
            batch_paths = patch_paths[i:i+batch_size]
            batch = []
            for p in batch_paths:
                img = Image.open(p).convert('RGB')
                batch.append(transform(img))

            batch_tensor = torch.stack(batch).to(device)
            features = model(batch_tensor).squeeze(-1).squeeze(-1)  # [B, 2048]
            all_features.append(features.cpu())

    return torch.cat(all_features, dim=0)  # [N, 2048]

In [ ]:
# test on first slide
test_slide = 'DHMC_0001'
features = extract_slide_features(test_slide)
print(f'{test_slide}: feature shape = {features.shape}')

# save as .pt file
torch.save(features, FEATURE_DIR / f'{test_slide}.pt')
print(f'Saved to features/{test_slide}.pt')

In [ ]:
# run feature extraction on all slides in patches/
slide_dirs = sorted([d for d in PATCH_DIR.iterdir() if d.is_dir()])
print(f'Found {len(slide_dirs)} slides to process')

for i, slide_dir in enumerate(slide_dirs):
    slide_name = slide_dir.name
    save_path = FEATURE_DIR / f'{slide_name}.pt'

    # skip if already done
    if save_path.exists():
        print(f'[{i+1}/{len(slide_dirs)}] {slide_name} already done, skipping')
        continue

    features = extract_slide_features(slide_name)
    if features is not None:
        torch.save(features, save_path)

    if (i + 1) % 10 == 0:
        print(f'[{i+1}/{len(slide_dirs)}] done')

print('Feature extraction complete')
print(f'Files saved in: {FEATURE_DIR}')

In [ ]:
# verify output
pt_files = sorted(FEATURE_DIR.glob('*.pt'))
print(f'Total .pt files: {len(pt_files)}')

# check one file
sample = torch.load(pt_files[0])
print(f'Sample: {pt_files[0].name}, shape: {sample.shape}')